# 07_graphgrued_ordered5_window_report

Summarize the **speed vs. accuracy trade-off** for W1/W2/W4.

In [ ]:

from pyspark.sql import functions as F

RESULT_TABLE = "dbo.tbl_graphgrued_ordered5_window_results"
res_df = spark.table(RESULT_TABLE)
print(res_df.count())
display(res_df)


In [ ]:

# best run per (window, imputer)
from pyspark.sql.window import Window

w = Window.partitionBy('obs_window_weeks', 'imputer_name').orderBy(
    F.col('test_macro_f1').desc(),
    F.col('test_label_D_f1').desc(),
    F.col('test_weighted_f1').desc(),
    F.col('runtime_sec').asc(),
)

best_per_window_imputer = (
    res_df
    .withColumn('rn', F.row_number().over(w))
    .filter(F.col('rn') == 1)
    .drop('rn')
    .orderBy('obs_window_weeks', 'imputer_name')
)

print('Best run per (window, imputer):')
display(best_per_window_imputer)


In [ ]:

# best overall per window
w2 = Window.partitionBy('obs_window_weeks').orderBy(
    F.col('test_macro_f1').desc(),
    F.col('test_label_D_f1').desc(),
    F.col('test_weighted_f1').desc(),
    F.col('runtime_sec').asc(),
)

best_per_window = (
    res_df
    .withColumn('rn', F.row_number().over(w2))
    .filter(F.col('rn') == 1)
    .drop('rn')
    .orderBy('obs_window_weeks')
)

print('Best overall per window:')
display(best_per_window)


In [ ]:

# compare each window against 4-week best
best_pd = best_per_window.toPandas()
base4 = best_pd[best_pd['obs_window_weeks'] == 4]
if len(base4) == 1:
    base4 = base4.iloc[0]
    comp = best_pd.copy()
    comp['delta_test_macro_f1_vs_4w'] = comp['test_macro_f1'] - base4['test_macro_f1']
    comp['delta_test_label_D_f1_vs_4w'] = comp['test_label_D_f1'] - base4['test_label_D_f1']
    comp['runtime_ratio_vs_4w'] = comp['runtime_sec'] / max(base4['runtime_sec'], 1e-9)
    display(spark.createDataFrame(comp))
else:
    print('Could not compute delta vs 4w baseline because 4w best run was not found uniquely.')
